# 📊 Неделя 2 — Анализ данных (EDA)
**Датасет:** Student Performance (UCI)  
**Цель:** Предсказать итоговую оценку студента (G3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Скачиваем датасет прямо здесь
import urllib.request, zipfile, os

os.makedirs('../data/raw', exist_ok=True)
if not os.path.exists('../data/raw/student-mat.csv'):
    print('Скачиваем датасет...')
    urllib.request.urlretrieve(
        'https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip',
        '../data/raw/student.zip'
    )
    with zipfile.ZipFile('../data/raw/student.zip') as z:
        z.extractall('../data/raw')
    os.remove('../data/raw/student.zip')
    print('Готово!')

df = pd.read_csv('../data/raw/student-mat.csv', sep=';')
print(f'Датасет загружен: {df.shape[0]} строк, {df.shape[1]} столбцов')

## 1. Общая информация о датасете

In [ ]:
print('=== РАЗМЕР ДАТАСЕТА ===')
print(f'Строк: {df.shape[0]}, Столбцов: {df.shape[1]}')
print()
print('=== ТИПЫ КОЛОНОК ===')
print(df.dtypes)
print()
print('=== ПРОПУСКИ ===')
print(df.isnull().sum())

In [ ]:
df.describe()

**Вывод:** Датасет содержит 395 студентов и 33 признака. Пропусков нет — данные чистые. Целевая переменная G3 (итоговая оценка) варьируется от 0 до 20.

## 2. График 1 — Распределение итоговой оценки G3

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['G3'], bins=20, color='steelblue', edgecolor='white')
plt.title('Распределение итоговой оценки (G3)', fontsize=14)
plt.xlabel('Итоговая оценка (G3)')
plt.ylabel('Количество студентов')
plt.axvline(df['G3'].mean(), color='red', linestyle='--', label=f'Среднее: {df["G3"].mean():.1f}')
plt.legend()
plt.tight_layout()
plt.show()

**Вывод:** Средняя оценка около 10.4 из 20. Заметен пик у нуля — это студенты которые не сдали или не пришли на экзамен. Большинство студентов получают оценки от 8 до 14.

## 3. График 2 — Оценка G3 по полу

In [ ]:
plt.figure(figsize=(8, 5))
df.groupby('sex')['G3'].mean().plot(kind='bar', color=['salmon', 'steelblue'], edgecolor='white')
plt.title('Средняя итоговая оценка по полу', fontsize=14)
plt.xlabel('Пол (F — девушки, M — парни)')
plt.ylabel('Средняя оценка G3')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Вывод:** Разница в оценках между девушками и парнями минимальная. Пол не является ключевым фактором успеваемости.

## 4. График 3 — Время на учёбу vs Оценка

In [ ]:
plt.figure(figsize=(8, 5))
df.groupby('studytime')['G3'].mean().plot(kind='bar', color='mediumseagreen', edgecolor='white')
plt.title('Среднняя оценка G3 в зависимости от времени на учёбу', fontsize=13)
plt.xlabel('Время на учёбу (1=<2ч, 2=2-5ч, 3=5-10ч, 4=>10ч)')
plt.ylabel('Средняя оценка G3')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Вывод:** Чем больше студент тратит времени на учёбу — тем выше его оценка. Это ожидаемая закономерность. Студенты занимающиеся более 10 часов в неделю получают в среднем на 2-3 балла больше.

## 5. График 4 — Пропуски занятий vs Оценка

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(df['absences'], df['G3'], alpha=0.4, color='coral')
plt.title('Зависимость оценки от количества пропусков', fontsize=14)
plt.xlabel('Количество пропусков')
plt.ylabel('Итоговая оценка G3')
plt.tight_layout()
plt.show()

**Вывод:** Прямой зависимости между пропусками и оценкой нет. Однако студенты с очень большим количеством пропусков (>40) редко получают высокие оценки.

## 6. График 5 — Образование родителей vs Оценка

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.groupby('Medu')['G3'].mean().plot(kind='bar', ax=axes[0], color='orchid', edgecolor='white')
axes[0].set_title('Оценка vs Образование матери', fontsize=12)
axes[0].set_xlabel('Уровень образования (0=нет, 4=высшее)')
axes[0].set_ylabel('Средняя оценка G3')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('Fedu')['G3'].mean().plot(kind='bar', ax=axes[1], color='skyblue', edgecolor='white')
axes[1].set_title('Оценка vs Образование отца', fontsize=12)
axes[1].set_xlabel('Уровень образования (0=нет, 4=высшее)')
axes[1].set_ylabel('Средняя оценка G3')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

**Вывод:** Чем выше образование родителей — тем лучше успеваемость студента. Дети родителей с высшим образованием получают в среднем на 2-3 балла выше. Образование семьи — важный фактор.

## 7. График 6 — Корреляционная матрица

In [ ]:
plt.figure(figsize=(12, 8))
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Корреляционная матрица числовых признаков', fontsize=14)
plt.tight_layout()
plt.show()

**Вывод:** Наибольшую корреляцию с G3 имеют G1 и G2 (оценки за 1-й и 2-й период) — это ожидаемо. Также заметна положительная корреляция с образованием родителей (Medu, Fedu) и отрицательная с количеством провалов (failures).

## 8. График 7 — Провалы в прошлом vs Оценка

In [ ]:
plt.figure(figsize=(8, 5))
df.groupby('failures')['G3'].mean().plot(kind='bar', color='tomato', edgecolor='white')
plt.title('Средняя оценка G3 в зависимости от количества провалов', fontsize=13)
plt.xlabel('Количество предыдущих провалов (0, 1, 2, 3+)')
plt.ylabel('Средняя оценка G3')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Вывод:** Это один из самых сильных факторов. Студенты без провалов в прошлом получают в среднем ~11 баллов, а с 3+ провалами — около 6. Количество провалов — важный предиктор.

## 9. График 8 — Оценки G1, G2, G3 в сравнении

In [ ]:
plt.figure(figsize=(10, 5))
plt.boxplot([df['G1'], df['G2'], df['G3']], labels=['G1 (1-й период)', 'G2 (2-й период)', 'G3 (итог)'])
plt.title('Сравнение оценок за все периоды', fontsize=14)
plt.xlabel('Период')
plt.ylabel('Оценка')
plt.tight_layout()
plt.show()

**Вывод:** Медианные оценки по всем трём периодам примерно одинаковы (~11). Однако в G3 больше выбросов снизу — часть студентов резко ухудшила результат к концу года. G1 и G2 хорошо предсказывают G3.

## 10. Итоговые выводы по анализу

**Ключевые наблюдения:**

1. **G1 и G2** — самые сильные предикторы итоговой оценки G3 (корреляция >0.8)
2. **Количество провалов (failures)** — сильно снижает итоговую оценку
3. **Время на учёбу (studytime)** — больше времени = выше оценка
4. **Образование родителей** — положительно влияет на успеваемость
5. **Пол** — практически не влияет на оценку
6. **Пропуски** — слабая отрицательная зависимость
7. Пропусков в данных **нет** — датасет чистый

**На следующей неделе:** обработаем категориальные признаки и обучим модели ML.